In [ ]:
# --- Cell 1: environment + scratch space discovery -------------------------
import os, sys, platform, subprocess

print("project:", "Edge0-120B")
print("python:", sys.version.split()[0])
print("platform:", platform.platform())

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for p in ("/kaggle/temp", "/kaggle/working"):
    os.makedirs(p, exist_ok=True)
    st = os.statvfs(p)
    free = st.f_bavail * st.f_frsize / 1e9
    print(f"{p}: free={free:.1f} GB writable={os.access(p, os.W_OK)}")

print("--- GPU ---")
print(subprocess.run(["nvidia-smi",
                      "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "no nvidia-smi")

print("--- internet ---")
for u in ("https://huggingface.co", "https://github.com", "https://gh-proxy.com"):
    r = subprocess.run(["curl", "-4", "-s", "-o", "/dev/null", "-w", "%{http_code}",
                        "--connect-timeout", "5", "--max-time", "10", u], capture_output=True, text=True)
    print(u, "->", r.stdout)


In [ ]:
# --- Cell 2: build Edge0 (strata + edge0-infer) ----------------------------
import os, subprocess, shutil, sys, glob, time

SRC = "/kaggle/temp/Edge0"
STRATA = os.path.join(SRC, "edge0-strata")
INFER = os.path.join(SRC, "edge0-infer")
LLAMA_COMMIT = "3cf03257f219afbe7334045ff7c6a06ac68c627d"

def run(cmd, cwd=None, env=None):
    print("+", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, env=env, check=True,
                          capture_output=False)

print("--- toolchain ---", flush=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)
nvcc = shutil.which("nvcc") or "/usr/local/cuda/bin/nvcc"
print("nvcc:", nvcc, subprocess.run([nvcc, "--version"], capture_output=True,
                                    text=True).stdout.strip().splitlines()[-1:])
run(["pip", "install", "-q", "cmake", "ninja", "uv"])

if not os.path.isdir(SRC):
    run(["git", "clone", "--depth", "1",
         "https://github.com/Edge0-AI/Edge0.git", SRC])

CACHE = sorted(glob.glob("/kaggle/input/*/edge0-engine"))
def maybe(cmd, **kw):
    if not CACHE:
        run(cmd, **kw)
if CACHE:
    for d in (os.path.join(STRATA, "build"), os.path.join(INFER, "build", "cpu", "bin")):
        os.makedirs(d, exist_ok=True)
    shutil.copy2(os.path.join(CACHE[0], "strata"), os.path.join(STRATA, "build", "strata"))
    shutil.copy2(os.path.join(CACHE[0], "edge0-infer"), os.path.join(INFER, "build", "cpu", "bin", "edge0-infer"))
    for so in glob.glob(os.path.join(CACHE[0], "*.so*")):
        shutil.copy2(so, os.path.join(STRATA, "build"))
    print("cached engine:", CACHE[0], flush=True)
LLAMA_DIR = os.path.join(STRATA, "third_party", "llama.cpp")
if not os.path.exists(os.path.join(LLAMA_DIR, "CMakeLists.txt")):
    os.makedirs(LLAMA_DIR, exist_ok=True)
    run(["bash", "-lc",
         "curl -L https://github.com/ggml-org/llama.cpp/archive/%s.tar.gz | "
         "tar -xz --strip-components=1 -C %s" % (LLAMA_COMMIT, LLAMA_DIR)])

env = dict(os.environ)
env["PATH"] = os.path.dirname(nvcc) + ":" + env["PATH"]
maybe(["cmake", "-S", STRATA, "-B", os.path.join(STRATA, "build"), "-G", "Ninja",
     "-DCMAKE_BUILD_TYPE=Release", "-DCMAKE_CUDA_COMPILER=" + nvcc,
     "-DCMAKE_CUDA_ARCHITECTURES=75", "-DSTRATA_ENABLE_CUDA=ON",
     "-DSTRATA_GGML_DIR=" + LLAMA_DIR], env=env)
maybe(["cmake", "--build", os.path.join(STRATA, "build"), "--target", "strata"], env=env)

maybe(["cmake", "-S", INFER, "-B", os.path.join(INFER, "build", "cpu"),
     "-G", "Ninja", "-DCMAKE_BUILD_TYPE=Release"], env=env)
maybe(["cmake", "--build", os.path.join(INFER, "build", "cpu"), "-j", "4"], env=env)

venv = os.path.join(STRATA, ".venv")
run(["uv", "venv", "--python", sys.executable, venv])
python = os.path.join(venv, "bin", "python")
run(["uv", "pip", "install", "--python", python,
     "-r", os.path.join(STRATA, "requirements.txt")])
if not CACHE:
    out = "/kaggle/working/edge0-engine"
    os.makedirs(out, exist_ok=True)
    shutil.copy2(os.path.join(STRATA, "build", "strata"), out)
    shutil.copy2(os.path.join(INFER, "build", "cpu", "bin", "edge0-infer"), out)
    for so in glob.glob(os.path.join(STRATA, "build", "*.so*")):
        shutil.copy2(so, out)
    print("cached engine ->", out, flush=True)
print("build done", flush=True)


In [ ]:
# --- Cell 3: download the pack, serve, publish tunnels ----------------------
import os, re, json, time, glob, subprocess, threading, http.server, socket, urllib.request

PORT = int("8000")
CTL_PORT = int("8081")
CTX = int("32768")
MODEL_REPO = "Edge0/Edge0-120B-q3k"
MODEL_ALIAS = "edge0-120b,edge0_120b"
STOP_REQUESTED = threading.Event()
WORK = "/kaggle/temp/run"
PACK = "/kaggle/temp/edge0-pack"
os.makedirs(WORK, exist_ok=True)

from huggingface_hub import snapshot_download
pack = snapshot_download(MODEL_REPO, local_dir=PACK, max_workers=8)
print("pack:", pack, flush=True)

CF = "/kaggle/temp/cloudflared"
if not os.path.exists(CF):
    subprocess.run(["curl", "-4", "-fSL", "--retry", "3", "-o", CF,
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                   check=True)
    os.chmod(CF, 0o755)

def start_tunnel(url, log_path):
    fh = open(log_path, "w")
    subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", url],
                     stdout=fh, stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(2)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log_path).read())
        if m:
            return m.group(0)
    return None

STOP_TOKEN = os.environ.get("CLUSTER_STOP_TOKEN") or os.urandom(8).hex()
class Ctl(http.server.BaseHTTPRequestHandler):
    def log_message(self, *a): pass
    def _send(self, code, obj):
        b = json.dumps(obj).encode()
        self.send_response(code); self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(b))); self.end_headers(); self.wfile.write(b)
    def do_GET(self):
        if self.path.rstrip("/") in ("", "/health"):
            self._send(200, {"ok": True, "stopping": STOP_REQUESTED.is_set()})
        elif self.path.startswith("/stop/" + STOP_TOKEN):
            STOP_REQUESTED.set(); self._send(200, {"ok": True})
        else:
            self._send(404, {"ok": False})
    def do_POST(self): self.do_GET()
ctl = http.server.ThreadingHTTPServer(("127.0.0.1", CTL_PORT), Ctl)
threading.Thread(target=ctl.serve_forever, daemon=True).start()

SRC = "/kaggle/temp/Edge0"
BIN = glob.glob(os.path.join(SRC, "edge0-infer", "build", "cpu", "bin", "edge0-infer"))[0]
LOG = os.path.join(WORK, "edge0.log")
SPLIT = os.environ.get("EDGE0_SPLIT", "1") == "1"
base = [BIN, "serve", "--model", pack, "--context", str(CTX)]
variants = []
if SPLIT:
    variants.append(base + ["--device", "cuda",
                            "--strata-arg", "--layer-split", "--strata-arg", "auto",
                            "--strata-arg", "--vram-reserve-mib", "--strata-arg", "700"])
variants.append(base + ["--device", "cuda:0"])
ready = False
server = None
for attempt, cmd in enumerate(variants, 1):
    print(f"launching (attempt {attempt}):", " ".join(cmd), flush=True)
    fh = open(LOG, "w")
    server = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT)
    for i in range(240):
        time.sleep(5)
        if server.poll() is not None:
            print(open(LOG).read()[-3000:], flush=True)
            break
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=4) as r:
                if r.status == 200:
                    ready = True
                    break
        except Exception:
            pass
        if i % 12 == 0:
            print(f"[{i*5}s] loading...", flush=True)
    if ready:
        break
    print(f"attempt {attempt} failed; trying the next launch variant", flush=True)
if not ready:
    print(open(LOG).read()[-4000:], flush=True)
    raise RuntimeError("edge0 did not become healthy")
print("server is UP on", PORT, flush=True)

public_url = start_tunnel(f"http://127.0.0.1:{PORT}", os.path.join(WORK, "cf.log"))
control_url = start_tunnel(f"http://127.0.0.1:{CTL_PORT}", os.path.join(WORK, "cf-ctl.log"))
if not public_url or not control_url:
    raise RuntimeError("could not obtain tunnel URLs")
open("/kaggle/working/CONTROL_URL.txt", "w").write(public_url + "\n")
open("/kaggle/working/STOP_URL.txt", "w").write(control_url + "/stop/" + STOP_TOKEN + "\n")
print("=" * 72)
print("CONTROL_URL=" + public_url)
print("STOP_URL=" + control_url + "/stop/" + STOP_TOKEN)
print("MODEL=" + MODEL_ALIAS)
print("=" * 72)


In [ ]:
# --- Cell 5: keep the tunnel alive (this is what you keep running) ---------
import time, urllib.request

PORT = int("8000")
SERVE_MINUTES = int("660")
AUTH_USER = "alamgir-zk"
AUTH_PASS = "Zaid2002@kaggle"

url = open("/kaggle/working/CONTROL_URL.txt").read().strip()
print(f"serving for ~{SERVE_MINUTES} minutes at {url}", flush=True)

for minute in range(SERVE_MINUTES):
    # Wait up to 60s, but poll the stop flag every 5s so shutdown is prompt.
    for _ in range(12):
        if STOP_REQUESTED.is_set():
            break
        time.sleep(5)
    if STOP_REQUESTED.is_set():
        print("STOP received from control tunnel -> ending session", flush=True)
        break
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/health")
        if AUTH_USER and AUTH_PASS:
            import base64
            auth_str = base64.b64encode(f"{AUTH_USER}:{AUTH_PASS}".encode()).decode()
            req.add_header("Authorization", f"Basic {auth_str}")
        with urllib.request.urlopen(req, timeout=5) as r:
            status = str(r.status)
    except Exception as e:
        status = f"err {e}"
    if minute % 5 == 0:
        # Re-print periodically so the URL can always be recovered from logs.
        print(f"[{minute}m] health={status} CONTROL_URL={url}", flush=True)
        try:
            gpu_stats = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu", "--format=csv,noheader"],
                                       capture_output=True, text=True).stdout.strip()
            for line in gpu_stats.splitlines():
                print(f"  [GPU {line.strip()}]", flush=True)
        except Exception:
            pass

print("SERVE_WINDOW_COMPLETE", flush=True)
